# 02. Fashion stores: blocking when the units are heterogeneous

**Sector:** fashion brick-and-mortar. **Decision:** does a new window layout
increase sales? The units are **120 stores**, and they differ a lot: a `G`
(large) store sells far more than a `PP` (tiny) one, regardless of layout. That
between-size variation, left uncontrolled, drowns the effect.

The answer is to **block by store size** (`PP`, `P`, `M`, `G`): randomize within
each block and combine by a weighted average. Theory:
[II. Designs](../../../docs/en/theory/02-designs.md) (topic 5) and
[III. Estimation](../../../docs/en/theory/03-estimation.md) (topic 9).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "fashion_stores.csv")
# Baseline sales differ strongly by size: this is the between-block variation.
print(df.groupby("store_size")["sales_y0"].mean().round(2))
df.head()


## 1. Blocked design and balance

We randomize within each size with `BlockedCRD(block_col="store_size")`. Right
after the draw (before attaching the outcome), we check covariate **balance**
with `check_balance`: randomization should leave treatment and control similar on
`foot_traffic_pre`. We assess it by the **magnitude of the SMD** (`|SMD| < 0.1`
is good), not by a p-value.


In [ ]:
from skxperiments.core.assignment import BlockedAssignment
from skxperiments.design.blocked_crd import BlockedCRD
from skxperiments.design.balance import check_balance

design = BlockedCRD(block_col="store_size", p=0.5, seed=202)
assignment = design.randomize(df[["region", "store_size", "foot_traffic_pre"]].copy())

# Balance is checked on the pre-treatment covariates only.
check_balance(assignment)[["covariate", "smd"]].round(3)


In [ ]:
from skxperiments.diagnostics import BalanceReport
from skxperiments.reporting import plot_balance

report = BalanceReport().run(assignment)
ax = plot_balance(report)
ax.set_title("Covariate balance (Love plot)")
ax.figure


## 2. Blocked estimate vs. ignoring the blocks

We attach the observed outcome and estimate with `BlockedDifferenceInMeans` (the
size-weighted average of the per-block effects) and `NeymanCI` (stratified
variance). To make the gain explicit, we compare with the **naive** analysis that
treats everything as a single `CRD`, ignoring store size: the standard error is
much larger, because the between-size variation enters the noise.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.blocked_difference_in_means import BlockedDifferenceInMeans
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["sales"] = np.where(t == 1, df["sales_y1"].to_numpy(), df["sales_y0"].to_numpy())

blocked_a = BlockedAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design,
    block_col=assignment.block_col_, block_sizes=assignment.block_sizes_, seed=202,
)
blocked_est = BlockedDifferenceInMeans(outcome_col="sales").fit(blocked_a)
blocked = NeymanCI(estimator=BlockedDifferenceInMeans(outcome_col="sales")).fit(blocked_a).estimate()

# Naive: same data, but analyzed as a single CRD (blocks ignored).
naive_a = CRDAssignment(data=data, treatment_col="treatment", design=CRD(p=0.5, seed=202), seed=202)
naive = NeymanCI(estimator=DifferenceInMeans(outcome_col="sales")).fit(naive_a).estimate()

print("per-block ATE:", {k: round(v, 3) for k, v in blocked_est.block_ates_.items()})
pd.DataFrame([
    {"analysis": "Blocked (correct)", "ATE": blocked.ate, "SE": blocked.se,
     "CI_low": blocked.ci[0], "CI_high": blocked.ci[1], "p": blocked.p_value},
    {"analysis": "Naive CRD (blocks ignored)", "ATE": naive.ate, "SE": naive.se,
     "CI_low": naive.ci[0], "CI_high": naive.ci[1], "p": naive.p_value},
]).round(3)


In [ ]:
from skxperiments.reporting import plot_effect

ax = plot_effect(blocked)
ax.set_title("Sales lift, blocked estimate (95% CI)")
ax.figure


## Decision

The true effect is `+0.40` (thousands in daily sales), constant across sizes.
The blocked estimate lands close to it and the interval excludes zero. Compare
the standard errors: the blocked analysis is **much more precise** than the naive
one, because it removed the between-size variation from the error. Same data
collection, firmer decision, just by respecting the structure of the design.

Next step: [03. Factorial campaign](03_fintech_factorial.ipynb).
